[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vinod-seth/Applied-Scientist-Interview-Gauntlet/blob/main/tutorial/06_dsa_algorithms/dsa_lab.ipynb)

# DSA Lab — Hidden Costs, Structures, and the Protocol Clock

**Session 6 armory notebook.** Every claim in Lessons 1–4 that can be *measured* is measured here rather than asserted. CPU-only, standard library plus NumPy and Matplotlib, no downloads, no network — the whole notebook runs in well under a minute.

▶ **[Open this notebook in Colab](https://colab.research.google.com/github/vinod-seth/Applied-Scientist-Interview-Gauntlet/blob/main/tutorial/06_dsa_algorithms/dsa_lab.ipynb)** (text link, in case the badge above does not render in your viewer)

| Part | Lesson | What it measures |
|---|---|---|
| 1 | 3 — Complexity | The container costs your complexity statement hides, timed |
| 2 | 3 — Complexity | Heap vs full sort vs quickselect for top-*k*, across *k* |
| 3 | 3 — Complexity | Three binary searches, one differential test, and which one breaks |
| 4 | 4 — Testing | Reservoir sampling really is uniform — checked against ±3 standard errors |
| 5 | 4 — Problem set | Union–find vs re-running BFS as edges stream in |
| 6 | 4 — Problem set | MinHash estimating Jaccard, and the 1/√m error law |
| 7 | 1 — Protocol | A beat timer for your own out-loud practice runs |

> **What to record:** nothing in this notebook belongs in your Metric Vault. Every number here is a *mechanism demonstration* on synthetic data — timings depend on this machine and this Python build. What belongs in your notes is the **sentence you would say in a round** ("membership on a list is a linear scan, so that loop is O(n·m), not O(n)"), never a timing from here.

**Environment.** Pinned below; **last verified 2026-07-29** on Python 3.11 and on Colab's default runtime. Runtimes shown in the output are indicative only.

In [ ]:
%pip install -q "numpy>=1.26" "matplotlib>=3.8"

In [ ]:
import time, random, heapq, zlib, sys
from collections import deque
import numpy as np
import matplotlib.pyplot as plt

random.seed(0)
np.random.seed(0)
plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

def timed(fn, *args, repeat=3, **kw):
    """Best-of-`repeat` wall time in milliseconds, plus the return value."""
    best, out = float("inf"), None
    for _ in range(repeat):
        t0 = time.perf_counter()
        out = fn(*args, **kw)
        best = min(best, time.perf_counter() - t0)
    return best * 1000.0, out

print("python", sys.version.split()[0], "| numpy", np.__version__)

---
## Part 1 — The costs your complexity statement hides

Lesson 3, Drill 1. `x in candidates` looks like a constant-time test. On a **list** it is a linear scan, so a loop containing it is $O(n \cdot m)$ and not $O(n)$ — and nothing in the source says so.

We time the same logical operation against a `list` and a `set`, and separately time string building by `+=` against `"".join`.

In [ ]:
def membership(container, probes):
    hits = 0
    for p in probes:
        if p in container:
            hits += 1
    return hits

sizes = [1_000, 2_000, 4_000, 8_000, 16_000]
probes_n = 2_000
list_ms, set_ms = [], []

print(f"{'container size':>14} | {'list (ms)':>10} | {'set (ms)':>9} | {'ratio':>7}")
print("-" * 50)
for m in sizes:
    data = list(range(m))
    probes = [random.randrange(2 * m) for _ in range(probes_n)]
    tl, hl = timed(membership, data, probes)
    ts, hs = timed(membership, set(data), probes)
    assert hl == hs, "the two containers must agree on the answer"
    list_ms.append(tl); set_ms.append(ts)
    print(f"{m:>14,} | {tl:>10.2f} | {ts:>9.2f} | {tl/ts:>6.0f}x")

In [ ]:
def build_with_concat(n):
    s = ""
    for i in range(n):
        s += "x"
    return len(s)

def build_with_join(n):
    parts = []
    for i in range(n):
        parts.append("x")
    return len("".join(parts))

for n in (20_000, 40_000, 80_000):
    tc, _ = timed(build_with_concat, n)
    tj, _ = timed(build_with_join, n)
    print(f"n={n:>7,}  +=: {tc:7.2f} ms   join: {tj:7.2f} ms   ratio: {tc/tj:5.1f}x")

print()
print("NOTE: CPython has an in-place optimisation for `str +=` when the string has")
print("a single reference, so this ratio is often modest here and catastrophic in")
print("code where the string is also referenced elsewhere. The interview answer is")
print("about the guarantee, not the optimisation: `join` is O(n), `+=` is not.")

In [ ]:
fig, ax = plt.subplots()
ax.plot(sizes, list_ms, "o-", label="`x in list` - linear scan")
ax.plot(sizes, set_ms, "s-", label="`x in set` - hash lookup")
ax.set_xlabel("container size (m), with 2,000 probes each")
ax.set_ylabel("time (ms)")
ax.set_title("The same line of code, two containers")
ax.legend()
plt.tight_layout(); plt.show()

**What you should have seen.** The set line is flat; the list line is a straight ramp. The ratio at the largest size is typically two to three orders of magnitude — and the *source code is identical*.

**The sentence to take into a round:** "Membership on a list is a linear scan, so that inner test makes the loop $O(n \cdot m)$ rather than $O(n)$ — I'll convert `candidates` to a set first, which makes it $O(n)$ on average."

---
## Part 2 — Heap vs sort vs quickselect for top-*k*

Lesson 3, Drill 3. Three ways to get the *k* largest of *n*, with three different bounds: $O(n \log k)$, $O(n \log n)$, $O(n)$ average. We check they agree, then time them as *k* grows.

In [ ]:
def topk_heap(a, k):
    """Size-k MIN-heap: the root is the admission threshold."""
    h = a[:k]
    heapq.heapify(h)
    for x in a[k:]:
        if x > h[0]:
            heapq.heapreplace(h, x)
    return sorted(h, reverse=True)

def topk_sort(a, k):
    return sorted(a, reverse=True)[:k]

def topk_quickselect(a, k):
    """Hoare 1961: partition, then recurse into ONE side only."""
    b = a[:]                      # quickselect reorders, so copy
    lo, hi, target = 0, len(b) - 1, k - 1
    while lo < hi:
        p = b[random.randint(lo, hi)]
        i, j = lo, hi
        while i <= j:             # three-way-ish Hoare partition, descending
            while b[i] > p: i += 1
            while b[j] < p: j -= 1
            if i <= j:
                b[i], b[j] = b[j], b[i]
                i += 1; j -= 1
        if target <= j:   hi = j
        elif target >= i: lo = i
        else:             break
    return sorted(b[:k], reverse=True)

n = 200_000
data = [random.randrange(10**9) for _ in range(n)]

for k in (1, 10, 1_000):
    assert topk_heap(data, k) == topk_sort(data, k) == topk_quickselect(data, k)
print("all three methods agree on the answer\n")

ks = [1, 10, 100, 1_000, 10_000, 100_000]
rows = []
print(f"{'k':>8} | {'heap O(n log k)':>16} | {'sort O(n log n)':>16} | {'quickselect O(n)':>17}")
print("-" * 68)
for k in ks:
    th, _ = timed(topk_heap, data, k)
    ts, _ = timed(topk_sort, data, k)
    tq, _ = timed(topk_quickselect, data, k)
    rows.append((k, th, ts, tq))
    print(f"{k:>8,} | {th:>16.1f} | {ts:>16.1f} | {tq:>17.1f}")

In [ ]:
ks_, th_, ts_, tq_ = zip(*rows)
fig, ax = plt.subplots()
ax.plot(ks_, th_, "o-", label="size-k min-heap  O(n log k)")
ax.plot(ks_, ts_, "s-", label="full sort  O(n log n)")
ax.plot(ks_, tq_, "^-", label="quickselect  O(n) average")
ax.set_xscale("log"); ax.set_xlabel("k (log scale)"); ax.set_ylabel("time (ms)")
ax.set_title(f"Top-k of n = {n:,}: the heap's advantage is a function of k")
ax.legend()
plt.tight_layout(); plt.show()

**What you should have seen.** The heap wins by a wide margin at small *k* and loses that margin as *k* approaches *n* — exactly what $O(n\log k)$ versus $O(n \log n)$ predicts. Quickselect is competitive throughout because its bound has no *k* in it at all.

**The three things the timing does not show, and which decide real answers:**

1. The heap needs **$O(k)$ memory**; the sort and quickselect both need the whole array.
2. The heap works on a **stream**. The other two need the data materialised — so at $n$ too large to hold, they are not slower, they are *unavailable*.
3. Quickselect **reorders your array** and has an $O(n^2)$ worst case on adversarial pivots. Random pivot choice (used above) is what keeps that unlikely.

---
## Part 3 — Three binary searches and one differential test

Lesson 3, Drill 5. Two of the implementations below are wrong in ways that pass a hand-picked example. We find out with a **differential test**: run every implementation against a brute-force reference on thousands of small random inputs, and report the smallest failing case.

This is the single highest-value testing habit in the whole session, and it takes ten lines.

In [ ]:
def bs_correct(a, t):
    lo, hi = 0, len(a) - 1
    while lo <= hi:
        mid = lo + (hi - lo) // 2      # overflow-safe form, by habit
        if a[mid] == t: return mid
        if a[mid] < t:  lo = mid + 1
        else:           hi = mid - 1
    return -1

def bs_offbyone(a, t):
    lo, hi = 0, len(a) - 1
    while lo < hi:                      # BUG: misses the last candidate
        mid = lo + (hi - lo) // 2
        if a[mid] == t: return mid
        if a[mid] < t:  lo = mid + 1
        else:           hi = mid - 1
    return -1

def bs_nonterminating(a, t, cap=64):
    """Boundary template with `lo = mid` and a rounding-down midpoint."""
    lo, hi, steps = 0, len(a) - 1, 0
    while lo < hi:
        steps += 1
        if steps > cap:
            raise RuntimeError("did not terminate: lo stopped moving")
        mid = lo + (hi - lo) // 2       # rounds DOWN, and the update below is lo = mid
        if a[mid] < t: lo = mid
        else:          hi = mid
    return lo if a and a[lo] == t else -1

def reference(a, t):
    for i, v in enumerate(a):
        if v == t: return i
    return -1

def differential_test(fn, trials=4000, max_n=8, max_v=6):
    for _ in range(trials):
        a = sorted(random.randrange(max_v) for _ in range(random.randrange(max_n + 1)))
        t = random.randrange(max_v)
        want = reference(a, t)
        try:
            got = fn(a, t)
        except RuntimeError as e:
            return {"input": (a, t), "verdict": f"raised: {e}"}
        # any index holding the target counts as correct
        ok = (want == -1 and got == -1) or (got != -1 and 0 <= got < len(a) and a[got] == t)
        if not ok:
            return {"input": (a, t), "expected": want, "got": got, "verdict": "wrong answer"}
    return None

for name, fn in [("bs_correct", bs_correct),
                 ("bs_offbyone", bs_offbyone),
                 ("bs_nonterminating", bs_nonterminating)]:
    fail = differential_test(fn)
    if fail is None:
        print(f"{name:>19}: PASS on 4,000 random cases")
    else:
        print(f"{name:>19}: FAIL  {fail}")

In [ ]:
def mid_overflow_32bit(lo, hi):
    """What `(lo + hi) / 2` does in a 32-bit signed integer type."""
    s = (lo + hi) & 0xFFFFFFFF
    if s >= 2**31: s -= 2**32          # wrap to signed
    return s // 2

lo, hi = 2**30, 2**31 - 1
print(f"lo = {lo:,}   hi = {hi:,}")
print(f"  (lo + hi) // 2 in 32-bit ints : {mid_overflow_32bit(lo, hi):,}   <- negative index")
print(f"  lo + (hi - lo) // 2           : {lo + (hi - lo)//2:,}   <- correct")
print()
print("Python integers are arbitrary-precision, so this cannot happen here - but the")
print("hazard is real in C++, Java and Go, and Bloch (2006) found it in the JDK after")
print("nine years. Knowing WHY it does not apply in Python is the stronger answer.")

**What you should have seen.** `bs_correct` passes; `bs_offbyone` fails on a small case (often a one- or two-element array); `bs_nonterminating` hits the step cap, which is what an infinite loop looks like when you guard it.

**The habit to keep:** when an implementation has boundaries, write the brute-force reference — it is three lines — and diff them on random inputs. Hand-picked cases test what you thought of; random cases test what you did not.

---
## Part 4 — Reservoir sampling really is uniform

Lesson 4, Drill 3. The claim is that every item of an unknown-length stream ends up in the sample with probability exactly $k/n$. That is a testable claim, so we test it — the same way you would test any sampler you shipped.

In [ ]:
def reservoir(stream, k, rng):
    res = []
    for i, x in enumerate(stream):
        if i < k:
            res.append(x)
        else:
            j = rng.randrange(i + 1)     # uniform in [0, i]
            if j < k:
                res[j] = x
    return res

n, k, trials = 50, 5, 40_000
rng = random.Random(7)
counts = np.zeros(n, dtype=int)
for _ in range(trials):
    for item in reservoir(range(n), k, rng):
        counts[item] += 1

freq = counts / trials
p = k / n
se = (p * (1 - p) / trials) ** 0.5
worst = np.max(np.abs(freq - p)) / se

print(f"stream length n = {n}, reservoir k = {k}, repetitions = {trials:,}")
print(f"expected selection frequency  p = k/n = {p:.4f}")
print(f"standard error at this many repetitions = {se:.5f}")
print(f"observed frequency range = [{freq.min():.4f}, {freq.max():.4f}]")
print(f"largest deviation = {worst:.2f} standard errors  ->  "
      f"{'consistent with uniform' if worst < 4 else 'INVESTIGATE'}")

In [ ]:
fig, ax = plt.subplots()
ax.bar(np.arange(n), freq, color="#4f46e5", alpha=.75, label="observed frequency")
ax.axhline(p, color="#059669", lw=2, label="expected k/n")
ax.axhline(p + 3*se, color="#d97706", ls="--", lw=1, label="+/- 3 standard errors")
ax.axhline(p - 3*se, color="#d97706", ls="--", lw=1)
ax.set_xlabel("position in the stream (item 0 arrives first)")
ax.set_ylabel("fraction of runs in which the item was sampled")
ax.set_title("Reservoir sampling: no position is favoured, including the first k")
ax.set_ylim(p - 8*se, p + 8*se)
ax.legend(loc="lower right")
plt.tight_layout(); plt.show()

**What you should have seen.** A flat bar chart hugging the $k/n$ line, with essentially every bar inside ±3 standard errors — *including the first k items*, which is where a naive implementation is biased.

**Break it yourself** (worth two minutes): change `rng.randrange(i + 1)` to `rng.randrange(i)` and re-run. The bias appears immediately as a visible tilt, and the deviation figure blows past 4 standard errors. That is what a broken sampler looks like — and it is invisible in any single run of the sampler.

---
## Part 5 — Union–find vs re-running BFS on a streaming edge list

Lesson 4, Drill 5. Both answer "how many components are there now?" correctly. Only one is sized for a query after *every* edge.

In [ ]:
class UnionFind:
    def __init__(self, n):
        self.parent = list(range(n))
        self.size = [1] * n
        self.components = n
    def find(self, x):
        while self.parent[x] != x:                 # path compression, iterative
            self.parent[x] = self.parent[self.parent[x]]
            x = self.parent[x]
        return x
    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra == rb:
            return False
        if self.size[ra] < self.size[rb]:          # union by size
            ra, rb = rb, ra
        self.parent[rb] = ra
        self.size[ra] += self.size[rb]
        self.components -= 1
        return True

def counts_unionfind(n, edges):
    uf = UnionFind(n)
    out = []
    for a, b in edges:
        uf.union(a, b)
        out.append(uf.components)
    return out

def counts_bfs_per_edge(n, edges):
    adj = [[] for _ in range(n)]
    out = []
    for a, b in edges:
        adj[a].append(b); adj[b].append(a)
        seen, comps = [False] * n, 0
        for s in range(n):                          # full traversal, every edge
            if seen[s]:
                continue
            comps += 1
            q = deque([s]); seen[s] = True
            while q:
                u = q.popleft()
                for v in adj[u]:
                    if not seen[v]:
                        seen[v] = True; q.append(v)
        out.append(comps)
    return out

nodes, n_edges = 1_200, 1_500
edges = [(random.randrange(nodes), random.randrange(nodes)) for _ in range(n_edges)]
t_uf, r_uf = timed(counts_unionfind, nodes, edges, repeat=1)
t_bfs, r_bfs = timed(counts_bfs_per_edge, nodes, edges, repeat=1)
assert r_uf == r_bfs, "both must report the identical component sequence"
print(f"identical answers on all {n_edges:,} prefixes: {r_uf[:5]} ... {r_uf[-1]}")
print(f"union-find      : {t_uf:9.1f} ms")
print(f"BFS per edge    : {t_bfs:9.1f} ms   ({t_bfs/t_uf:.0f}x slower)")

In [ ]:
scales = [150, 300, 600, 1_200]
uf_t, bfs_t = [], []
for s in scales:
    e = [(random.randrange(s), random.randrange(s)) for _ in range(s + s // 4)]
    a, _ = timed(counts_unionfind, s, e, repeat=1)
    b, _ = timed(counts_bfs_per_edge, s, e, repeat=1)
    uf_t.append(a); bfs_t.append(b)

fig, ax = plt.subplots()
ax.plot(scales, uf_t, "o-", label="union-find  ~O(E alpha(n))")
ax.plot(scales, bfs_t, "s-", label="BFS re-run per edge  O(E(V+E))")
ax.set_yscale("log"); ax.set_xlabel("nodes (edges scale with nodes)")
ax.set_ylabel("time (ms, log scale)")
ax.set_title("Same answer, different asymptotics")
ax.legend()
plt.tight_layout(); plt.show()

**What you should have seen.** Identical component sequences, and a gap that widens with scale rather than staying constant — the signature of a *different complexity class*, not a slower constant.

**The point of the drill:** BFS is not wrong here, it is wrongly **sized** for the query pattern. Recognising "correct but wrongly sized" is a distinct skill from recognising "incorrect", and interviewers probe it deliberately.

---
## Part 6 — MinHash estimates Jaccard, and the error follows 1/√m

Lesson 4, Drill 4. The claim is exact: $\Pr[\min h(A) = \min h(B)] = J(A,B)$. With $m$ independent hash functions the estimate is a mean of $m$ Bernoulli draws, so its standard error should fall as $1/\sqrt{m}$. We measure both.

In [ ]:
MERSENNE = (1 << 61) - 1

def shingles(text, w=5):
    toks = text.split()
    return {" ".join(toks[i:i+w]) for i in range(max(0, len(toks) - w + 1))}

def base_hash(s):
    return zlib.crc32(s.encode("utf-8")) & 0xFFFFFFFF

def minhash_signature(base, coeffs):
    """base: the crc32 values of a document's shingles, hashed once."""
    return [min((a * h + b) % MERSENNE for h in base) for a, b in coeffs]

def jaccard(a, b):
    return len(a & b) / len(a | b) if (a | b) else 0.0

vocab = [f"w{i}" for i in range(400)]
rng = random.Random(11)
doc_a = " ".join(rng.choice(vocab) for _ in range(600))
tokens = doc_a.split()
doc_b = " ".join(t if rng.random() > 0.18 else rng.choice(vocab) for t in tokens)

A, B = shingles(doc_a), shingles(doc_b)
exact = jaccard(A, B)
hA = [base_hash(s) for s in A]          # hash each shingle once
hB = [base_hash(s) for s in B]
print(f"|A| = {len(A):,} shingles, |B| = {len(B):,} shingles")
print(f"exact Jaccard similarity = {exact:.4f}\n")

ms = [16, 64, 256, 1024]
rmse, theory = [], []
print(f"{'m':>6} | {'mean estimate':>14} | {'RMSE':>8} | {'sqrt(J(1-J)/m)':>15}")
print("-" * 54)
for m in ms:
    errs = []
    for trial in range(40):
        r = random.Random(1000 + trial)
        coeffs = [(r.randrange(1, MERSENNE), r.randrange(MERSENNE)) for _ in range(m)]
        sa, sb = minhash_signature(hA, coeffs), minhash_signature(hB, coeffs)
        est = sum(x == y for x, y in zip(sa, sb)) / m
        errs.append(est - exact)
    errs = np.array(errs)
    e = float(np.sqrt((errs ** 2).mean()))
    t = float(np.sqrt(exact * (1 - exact) / m))
    rmse.append(e); theory.append(t)
    print(f"{m:>6} | {exact + errs.mean():>14.4f} | {e:>8.4f} | {t:>15.4f}")

In [ ]:
fig, ax = plt.subplots()
ax.plot(ms, rmse, "o-", label="measured RMSE of the MinHash estimate")
ax.plot(ms, theory, "s--", label=r"theory: $\sqrt{J(1-J)/m}$")
ax.set_xscale("log", base=2); ax.set_yscale("log")
ax.set_xlabel("m (number of hash functions in the signature)")
ax.set_ylabel("error (log scale)")
ax.set_title("Signature length buys accuracy at the square-root rate")
ax.legend()
plt.tight_layout(); plt.show()

**What you should have seen.** The mean estimate sitting on the exact Jaccard value — MinHash is essentially unbiased — and the error falling at the $m^{-1/2}$ rate, a straight line of slope $-\tfrac12$ on log–log axes.

The measured curve runs **parallel to but slightly above** the idealised $\sqrt{J(1-J)/m}$ line, typically by 20–30%. That gap is real and worth understanding rather than tuning away: the theoretical expression assumes $m$ *independent* permutations, while the affine family $(ah + b) \bmod p$ used here is only pairwise independent, so the estimator carries a little more variance than the ideal. The *rate* is what the design decision rests on, and the rate is exactly as predicted.

**The interview consequence:** quadrupling the signature length halves the error, so a 128-value signature is a deliberate accuracy/memory operating point, not an arbitrary default. That is also the memory arithmetic to say out loud: 128 four-byte values across $10^8$ documents is about **51 GB**, which is why the pipeline shards.

---
## Part 7 — The protocol clock

Lesson 1. The beats are easy at leisure and collapse under a clock, so measure them. Run this while you solve a problem **out loud**: call `beat()` as you complete each one, and read the report afterwards.

The cell below runs a simulated session so the notebook executes end-to-end. Your real run is the commented block at the bottom.

In [ ]:
class ProtocolClock:
    BEATS = ["clarify", "approach", "narrate", "test", "optimize"]

    def __init__(self, budget_min=45):
        self.t0 = time.perf_counter()
        self.budget = budget_min * 60
        self.marks = {}

    def beat(self, name):
        assert name in self.BEATS, f"unknown beat: {name}"
        self.marks[name] = time.perf_counter() - self.t0
        print(f"  [{self.marks[name]/60:5.1f} min] {name} complete")

    def report(self):
        print("\n" + "=" * 58)
        print(f"{'beat':>10} | {'at (min)':>9} | {'took (min)':>11} | status")
        print("-" * 58)
        prev = 0.0
        for b in self.BEATS:
            if b in self.marks:
                at = self.marks[b]
                print(f"{b:>10} | {at/60:>9.1f} | {(at-prev)/60:>11.1f} | ok")
                prev = at
            else:
                print(f"{b:>10} | {'--':>9} | {'--':>11} | MISSING - no evidence produced")
        total = time.perf_counter() - self.t0
        missing = [b for b in self.BEATS if b not in self.marks]
        print("-" * 58)
        print(f"total: {total/60:.1f} min of a {self.budget/60:.0f} min budget")
        if missing:
            print(f"beats with no evidence: {', '.join(missing)}")
            print("A skipped beat is a paragraph the interviewer cannot write.")
        else:
            print("all five beats produced evidence.")
        if "approach" in self.marks and self.marks["approach"] < 60:
            print("WARNING: approach stated inside the first minute - did you clarify first?")

# --- simulated session so this notebook runs end to end ---------------------
demo = ProtocolClock(budget_min=45)
for b, pause in [("clarify", .05), ("approach", .05), ("narrate", .1), ("test", .05)]:
    time.sleep(pause)
    demo.beat(b)
demo.report()      # note: 'optimize' was never called, on purpose

In [ ]:
# ---------------------------------------------------------------------------
# YOUR RUN. Uncomment, start a real problem, and speak every beat out loud.
#
#   clock = ProtocolClock(budget_min=45)
#   ... restate the problem, ask 2-3 clarifying questions ...
#   clock.beat("clarify")
#   ... pattern + structure + complexity + why, then STOP and wait ...
#   clock.beat("approach")
#   ... code while narrating the invariant ...
#   clock.beat("narrate")
#   ... hand-trace base, boundary and inversion cases ...
#   clock.beat("test")
#   ... state the improvement with its cost, and OFFER it ...
#   clock.beat("optimize")
#   clock.report()
# ---------------------------------------------------------------------------
print("Targets for a 45-minute round: clarify by ~3 min, approach by ~8 min,")
print("narrate done by ~28 min, test done by ~38 min, optimize discussed by ~43 min.")

---
## What to take from this notebook

| Part | The sentence it earns you |
|---|---|
| 1 | "Membership on a list is a linear scan — that loop is $O(n \cdot m)$, not $O(n)$." |
| 2 | "A size-*k* min-heap is $O(n \log k)$ with $O(k)$ memory, and unlike a sort it works on a stream." |
| 3 | "I'd diff it against a brute-force reference on random inputs — hand-picked cases test what I thought of." |
| 4 | "Reservoir sampling gives every item probability $k/n$ in one pass; I'd verify it against $\pm 3$ standard errors." |
| 5 | "BFS isn't wrong here, it's wrongly sized — re-running it per edge is $O(E(V+E))$." |
| 6 | "MinHash is unbiased with error $\sqrt{J(1-J)/m}$, so 128 values is an operating point, not a default." |
| 7 | "I rehearse the five beats on a timer." |

> **Reminder:** none of the timings above are yours to quote. They are this machine's numbers on synthetic data. What transfers is the mechanism and the sentence — and that is exactly what the round scores.

**Back to:** [Session 6 README](README.md) · [Mock Round](05_mock_round.md) · [Chapter Quiz](quiz.md)